In [ ]:
import os
import json
import torch
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# Same cache dir as training so we reuse the downloaded base model
CACHE_DIR = "/projects/prjs1308/huggingface/"

# Path to the fine-tuned adapter + tokenizer
ADAPTER_DIR = "/projects/prjs1308/africa_llm_data/results/inference_models/20260308_205449_simple_gemma3/simple_gemma3_4b_lr0.0001_seed42_20260308_205449"

# Load run-time config saved during fine-tuning
with open(os.path.join(ADAPTER_DIR, "run_config.json"), "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

base_model_id = run_cfg["model_id"]
system_prompt = run_cfg.get("system_prompt")
prompt_template = run_cfg["prompt_template"]
max_tokens = run_cfg.get("max_tokens", 4096)
default_max_new_tokens = run_cfg.get("max_new_tokens", 500)
targets_spec = run_cfg.get("targets_spec")

# Device and quantization config (match training as closely as possible)
device = "cuda" if torch.cuda.is_available() else "cpu"
compute_dtype = (
    torch.bfloat16
    if device == "cuda" and torch.cuda.is_bf16_supported()
    else torch.float16
)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

# Load tokenizer/processor from adapter directory (use shared cache dir)
processor = AutoProcessor.from_pretrained(ADAPTER_DIR, cache_dir=CACHE_DIR)
tokenizer = getattr(processor, "tokenizer", processor)

# Load base Gemma-3 model and attach LoRA adapter
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=quant_config,
    device_map={"": 0} if device == "cuda" else None,
    torch_dtype=compute_dtype if device == "cuda" else None,
    cache_dir=CACHE_DIR,
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
model.to(device)

print("Loaded base model:", base_model_id)
print("Using adapter from:", ADAPTER_DIR)
print("Device:", device)

In [ ]:
import sys, os
import time
import re

# Ensure project root is importable so we can use agent_utils
PROJECT_ROOT = "/home/fcool/africa_llm"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from agent_utils.gemma3_finetune_simple import _extract_pred_json


def build_chat_input(text: str) -> str:
    """Build chat-formatted prompt using the same template + system prompt as training."""
    instruction = prompt_template.format(text)
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": instruction})
    chat = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    return chat


def generate_annotation(text: str, max_new_tokens: int | None = None):
    """Generate JSON annotation for a single post.

    Returns (raw_text, parsed_json, elapsed_sec).
    Uses _extract_pred_json first; if that fails, falls back to a
    loose key/value regex extractor so we still recover partial
    annotations when the JSON is slightly malformed/truncated.
    """
    if max_new_tokens is None:
        max_new_tokens = default_max_new_tokens

    start = time.time()

    chat = build_chat_input(text)
    inputs = tokenizer(chat, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Slice off the prompt tokens to get only the generated completion
    gen_ids = output_ids[0, inputs["input_ids"].shape[1]:]
    completion = tokenizer.decode(gen_ids, skip_special_tokens=True)

    # First, try the robust JSON extractor from training
    parsed = _extract_pred_json(completion)

    # Fallback: loose regex over "key": value pairs across the text
    if parsed is None:
        pattern = r'"([A-Za-z0-9_]+)"\s*:\s*(null|true|false|-?\d+\.\d+|-?\d+|"(?:[^"\\]|\\.)*")'
        results = {}
        for m in re.finditer(pattern, completion):
            key = m.group(1)
            raw_val = m.group(2)
            if raw_val == "null":
                val = None
            elif raw_val in ("true", "false"):
                val = 1 if raw_val == "true" else 0
            elif raw_val.startswith('"') and raw_val.endswith('"'):
                try:
                    val = json.loads(raw_val)
                except Exception:
                    val = raw_val.strip('"')
            else:
                try:
                    val = float(raw_val) if "." in raw_val else int(raw_val)
                except Exception:
                    val = raw_val
            results[key] = val
        parsed = results or None

    elapsed = time.time() - start
    return completion, parsed, elapsed


# Quick smoke test (replace with a real post)
example_text = "We are moving forward. We are not gonna slow down. We are fixing this country."
raw_out, parsed_out, t_sec = generate_annotation(example_text)
print("RAW COMPLETION:\n", raw_out)
print("\nPARSED JSON:\n", parsed_out)
print(f"\nElapsed: {t_sec:.2f}s")

In [ ]:
import pandas as pd
import os
import math

# Path to inference CSV (you control this)
DATA_PATH = "/home/fcool/africa_llm/inference/results/test_results/inference_data.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded {len(df)} rows from {DATA_PATH}")
else:
    print(f"CSV not found at {DATA_PATH}. No inference will run until this file exists.")
    df = pd.DataFrame(columns=["id", "text"])  # empty placeholder so the rest of the cell is safe

# Select the slice of rows to run inference on
# Option 1: explicit index range
start_idx = 0   # inclusive
end_idx = 10    # exclusive

# Option 2: pick a quarter of the dataframe
use_quarter = False   # set True to use quarter selection instead of explicit range
quarter = 1           # 1, 2, 3, or 4

if use_quarter:
    N = len(df)
    chunk_size = math.ceil(N / 4)
    start_idx = (quarter - 1) * chunk_size
    end_idx = min(quarter * chunk_size, N)

# Human-readable name for this inference range (for tracking in the CSV)
if use_quarter:
    range_name = f"quarter_{quarter}"
else:
    range_name = f"indices_{start_idx}_{end_idx}"

subset = df.iloc[start_idx:end_idx].copy()
print(f"Running inference on rows [{start_idx}, {end_idx}) → {len(subset)} examples (range={range_name})")

inference_times = []
records = []
n_seen = 0
n_ok = 0

# Output CSV path (we append to this file after each inference)
out_path = "data/inference_predictions.csv"

for idx, row in subset.iterrows():
    text = row["text"]
    row_id = row["id"]

    raw_out, parsed_out, t_sec = generate_annotation(text)
    inference_times.append(t_sec)
    avg_time = sum(inference_times) / len(inference_times)

    n_seen += 1
    json_ok = isinstance(parsed_out, dict)
    if json_ok:
        n_ok += 1
    success_pct = (n_ok / n_seen) * 100.0

    # Collect record for CSV: always keep id + json_ok + range_name, and as many parsed fields as we have
    rec = {"id": row_id, "json_ok": json_ok, "range_name": range_name}
    if isinstance(parsed_out, dict):
        rec.update(parsed_out)
    records.append(rec)

    print("\n" + "=" * 80)
    print(f"ROW {idx} (id={row_id})")
    print("TEXT:")
    print(text)
    print("\nRAW COMPLETION:")
    print(raw_out)
    print("\nPARSED JSON:")
    print(parsed_out)
    print(f"\nInference time: {t_sec:.2f}s  |  Running avg: {avg_time:.2f}s")
    print(f"JSON extracted: {json_ok}  |  Running success: {success_pct:.1f}%")

    # Append this record to CSV immediately so progress is not lost on crash
    row_df = pd.DataFrame([rec])
    write_header = not os.path.exists(out_path)
    row_df.to_csv(out_path, mode="a", index=False, header=write_header)

print(f"\nSaved predictions for {len(records)} rows (appended) to {out_path}")